# Explorando o LangGraph

Até agora trabalhamos com cadeias de execução lineares, em que o fluxo é fixo e previsível: o dado entra, passa por um modelo e gera uma saída. Agora, vamos mudar para uma arquitetura não linear, que passa a ser baseada em grafos. Vamos começar a explorar o **LangGraph**, uma ferramenta fundamental para a criação de aplicações que não dependem apenas de cadeias lineares, e são capazes de tomar decisões sobre qual caminho seguir ou qual especialização utilizar em tempo de execução.

Com o LangGraph é possível ter:

- Execução paralela: o sistema pode processar informações de formas distintas simultaneamente.
- Tomada de decisão dinâmica: o fluxo de execução pode ser alterado conforme a entrada do usuário ou o contexto, permitindo que a aplicação escolha um assistente especializado que é mais adequado para a tarefa específica naquele momento.
- Gerenciamento avançado de estado e rotas: em vez de um fluxo rígido, temos um grafo onde é possível rotear a consulta para diferentes caminhos antes de chegar a uma resposta final, trazendo uma modularidade muito maior para aplicações que exigem respostas mais precisas e contextualizadas.

Considerando o agente "assistente de viagens", com o LangGraph, esse agente pode ser composto por múltiplos _prompts_ especializados, como por exemplo. Com essa arquitetura, o agente pode direcionar o fluxo de processamento conforme a necessidade, escolhendo o _prompt_ especializado mais adequado para executar uma tarefa e retornar a resposta.

Para ilustrar esse novo paradigma, vamos implementar um fluxo não linear em que temos dois _prompts_ especializados: um no tema **praias** e outro em **montanhas**. A partir da pergunta do usuário, será definido qual _prompt_ usar para responder a pergunta.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
api_key       = os.getenv("OPENAI_API_KEY")
model_version = os.getenv("OPENAI_MODEL_VERSION")


llm = ChatOpenAI(
    model=model_version,
    api_key=api_key
)


# define template dos prompts especializados
prompt_beach = ChatPromptTemplate.from_messages([
    ("system",      "Apresente-se como Sra Praia. Você é uma especialista em viagens com destinos para praia."),
    ("placeholder", "{conversation_historic}"),
    ("human",       "{query}")
])

prompt_mountain = ChatPromptTemplate.from_messages([
    ("system",      "Apresente-se como Sr Montanha. Você é uma especialista em viagens com destinos para montanhas e atividades radicais."),
    ("placeholder", "{conversation_historic}"),
    ("human",       "{query}")
])


# define cadeias especializadas
chain_beach    = prompt_beach    | llm | StrOutputParser()
chain_mountain = prompt_mountain | llm | StrOutputParser()

## Implementando fluxo não linear manualmente

Primeiro vamos implementar um fluxo não linear "na mão", sem utilizar o LangGraph. Esse fluxo precisa começar por um "_prompt_ roteador", que recebe a pergunta do usuário e decide qual o caminho mais adequado, qual a melhor cadeia para receber e lidar com essa pergunta.

Para garantir que a resposta da "cadeia roteadora" seja algo que permita definir o melhor caminho a ser seguido, poderíamos usar um _output parser_ que tranforme o formato da resposta da forma mais adequada, mas vamos fazer um pouco diferente do que já foi implementado antes nesse projeto, vamos definir uma classe `Route` para representar a estrutura do dado que esperamos receber como resposta:

In [2]:
from typing import TypedDict, Literal

# define template do prompt roteador
prompt_router = ChatPromptTemplate.from_messages([
    ("system", "Responda apenas com 'praia' ou 'montanha'"),
    ("human", "{query}")
])

# define estrutura da resposta esperada para o prompt roteador
class Route(TypedDict):
    destination: Literal["praia", "montanha"]

Então definimos a cadeia que será responsável por tratar a pergunta inicial do usuário, passando o _prompt_ roteador e encadeando a classe `Route` no final, como já foi feito anteriormente com os _outout parsers_:

In [3]:
# define cadeia roteadora
chain_router   = prompt_router | llm | Route
response_route = chain_router.invoke({"query": "Quero surfar em um lugar quente"})

print(type(response_route))
response_route

<class 'dict'>


{'content': 'praia',
 'additional_kwargs': {'refusal': None},
 'response_metadata': {'token_usage': {'completion_tokens': 30,
   'prompt_tokens': 32,
   'total_tokens': 62,
   'completion_tokens_details': {'accepted_prediction_tokens': 0,
    'audio_tokens': 0,
    'reasoning_tokens': 19,
    'rejected_prediction_tokens': 0,
    'text_tokens': None},
   'prompt_tokens_details': {'audio_tokens': 0,
    'cache_write_tokens': 0,
    'cached_tokens': 0,
    'image_tokens': None,
    'text_tokens': None}},
  'model_provider': 'openai',
  'model_name': 'gpt-6-luna',
  'system_fingerprint': None,
  'id': 'chatcmpl-Ecyue6FbnLL7Gyw4bK60Pxzh6OnzI',
  'service_tier': 'default',
  'finish_reason': 'stop',
  'logprobs': None},
 'type': 'ai',
 'name': None,
 'id': 'lc_run--01a174b4-0f15-721b-ad27-1437de472f18-0',
 'tool_calls': [],
 'invalid_tool_calls': [],
 'usage_metadata': {'input_tokens': 32,
  'output_tokens': 30,
  'total_tokens': 62,
  'input_token_details': {'audio': 0, 'cache_read': 0, 'ca

Note que ao chamar `invoke` na cadeia, recebemos como retorno um objeto composto. Isso acontece porque a classe `Route` é apenas o **_schema_** (contrato) que descreve o formato do dado desejado, ela não implementa a interface de um `Runnable` ou de um `OutputParser`.

Ao usar `|` para encadear um objeto como componente da cadeia (logo após `llm`), é esperado que esse objeto possua um método chamado `invoke` (ou `parse`), que saibe exatamente como acessar a mensagem que o modelo de linguagem gerou e extrair apenas o conteúdo em texto (chave `content`). E a classe `Route` é apenas uma estrutura de dados (um `TypedDict` ou uma classe vazia), não possuindo os métodos necessários para processar o fluxo do LangChain. 

Então precisamos mudar a forma como definimos a cadeia, vamos usar o método [`with_structured_output`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/with_structured_output) no modelo de linguagem, que vai receber a clase `Route` como um modelo para criar um _parser_ customizado que receberá a resposta do modelo de linguagem.

O método `with_structured_output(Route)`, faz três coisas importantes que o simples encadeamento (`|`) não faz:

- configura o modelo (via parâmetros da API, como o _Function Calling_ ou _JSON Mode_) para que o modelo force a saída em um formato estruturado.
- injeta as definições de _schema_ (a classe `Route`) no contexto do modelo para que ele entenda exatamente o formato da resposta esperada.
- configura um _parser_ automático nos bastidores, e quando o modelo retorna a resposta, esse _parser_ pega o JSON/dado estruturado que veio da API e o converte para o formato definido na classe `Route`.

Seguindo essa abordagem, não é necessário instanciar manualmente um _parser_, e a cadeia usa a definição de `Route` para "moldar" a saída do modelo, convertendo a resposta em um dicionário Python que respeita as restrições impostas.

Se não usamos o `with_structured_output`, o modelo devolve apenas texto puro. E se quiséssemos usar a classe `Route` como um _parser_ sem o `with_structured_output`, seria necessário definir a classe herdando de `BaseOutputParser` e implementar o método `parse` para converter manualmente para um dicionário a _string_ resultante do modelo de linguagem.

In [4]:
from langchain_core.globals import set_debug

set_debug(True)
set_debug(False)

chain_router   = prompt_router | llm.with_structured_output(Route)
response_route = chain_router.invoke({"query": "Quero surfar em um lugar quente"})

print(type(response_route))
response_route

<class 'dict'>


{'destination': 'praia'}

E por fim criamos uma função que encapsula essa dinâmica, recebendo a pergunta inicial do usuário. Note que como fizemos a resposta da cadeia roteadora se adequar a definição da classe `Route`, podemos acessar a chave `'destination'` na resposta do método `invoke`.

In [5]:
def manual_response(user_query: str):
    route = chain_router.invoke({"query": user_query})["destination"]
    if route == "praia":
        return chain_beach.invoke({"query": user_query})
    return chain_mountain.invoke({"query": user_query})

In [6]:
print(manual_response("Quero passear em uma praia no Brasil"))

Oi! Sou a **Sra. Praia**, especialista em viagens de praia pelo Brasil. 🌴

Para começar, aqui vão três estilos de passeio:

- **Águas mornas e piscinas naturais:** Porto de Galinhas (PE)
- **Praias e trilhas em meio à natureza:** Ilha Grande (RJ)
- **Variedade de praias e boa estrutura:** Florianópolis (SC)

Você prefere uma praia mais tranquila ou com movimento? E de qual cidade pretende sair?


In [7]:
print(manual_response("Quero fazer um passeio radical na natureza"))

Olá! Sou o **Sr. Montanha**, especialista em viagens de natureza e aventura. 🏔️

Vamos encontrar o passeio radical ideal para você! **Em que cidade ou região você está** e que tipo de aventura prefere: trilha desafiadora, rafting, escalada, rapel, tirolesa ou outra? Se puder, diga também seu nível de experiência e quando pretende ir — assim sugiro opções que combinem com você e com as condições do lugar.


In [8]:
print(manual_response("Recentemente escalei o Pico do Paraná e agora quero mais experiências similares"))

Olá! Sou o **Sr. Montanha**, especialista em viagens de montanha e aventuras radicais. Depois do Pico do Paraná, você já tem uma boa base para encarar trilhas longas e exigentes. Estas podem ser ótimas próximas experiências:

- **Serra Fina (MG/SP/RJ)** — travessia de vários dias, com acampamento e subidas intensas; inclui a Pedra da Mina, um dos pontos mais altos do Brasil.
- **Marins–Itaguaré (SP/MG)** — percurso puxado, com trechos íngremes e expostos. Boa opção para quem quer mais navegação e aventura.
- **Pico das Agulhas Negras (RJ)** — uma subida mais técnica, com escalaminhadas e passagens que podem exigir equipamento ou acompanhamento de guia.
- **Travessia Petrópolis–Teresópolis (RJ)** — caminhada de montanha de vários dias, com paisagens espetaculares e trechos que pedem atenção, especialmente em condições ruins.
- **Monte Roraima** — expedição mais longa e remota, com logística especial; é uma grande mudança de escala e costuma ser feita com equipe local.

Para uma progress

## LangGraph

### O que é o LangGraph

O [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) tem como propósito principal transformar assistentes e fluxos de trabalho em **nós de um grafo**. Em vez de ter um código linear onde uma instrução segue a outra de forma rígida, criamos um grafo onde cada parte da aplicação (como o "roteador" ou o "gerador de resposta específica") é um nó nesse grafo. Ele permite definir regras claras para que o sistema navegue entre esses nós de acordo com as condições atingidas, como decidir entre uma resposta referente a praia ou montanha.

Seu funcionamento se baseia em uma estrutura de **estados**, onde cada execução carrega um conjunto de informações (como a pergunta inicial, o destino escolhido e a resposta final). Nós definimos esse estado através de uma classe, como fizemos com a classe `Route` extendendo `TypedDict`, garantindo que o sistema saiba exatamente o que está sendo processado em cada etapa. O LangGraph gerencia a transição entre esses nós, podendo ir do início ao destino através de condicionais.

Dentre algumas vantagens de utilizar o LangGraph podemos citar:

- Assincronicidade: o LangGraph permite criar métodos assíncronos, o que faz com que a aplicação não fique "travada" aguardando uma resposta única. Isso torna o sistema mais eficiente, especialmente quando lidamos com modelos de linguagem onde o tempo de resposta pode variar.

- Organização e modularidade: ao dividir o código em nós, fica muito mais fácil dar manutenção e entender o fluxo da aplicação. É possível isolar a lógica de roteamento da lógica de geração de resposta de cada assistente específico.

- flexibilidade no controle de fluxo: Através do uso de arestas condicionais (_conditional edges_), podemos ter um controle mais fino e profissional sobre para onde o dado deve ir após ser processado em um nó, o que é bem mais robusto do que estruturas de controle simples que implementamos manualmente.

### Implementando fluxo não linear com LangGraph

Vamos substituir a lógica de roteamento manual e sequencial implementada na função `manual_response` por uma estrutura baseada em grafos, que será formada por um conjunto de funções assíncronas que funcionam como os nós do grafo. Como não temos como saber quanto tempo a API de IA generativa vai levar para retornar sua resposta, precisamos instruir o Python a executar essas tarefas de forma assíncrona, para que o programa não fica bloqueado esperando a resposta da API, liberando o processamento para outras tarefas e tornando o sistema mais eficiente e rápido.

As funções que serão implementadas são:

- "nó roteador": é o ponto de entrada, recebe o "estado", aciona o roteador para identificar o destino desejado pelo usuário e atualiza o estado com essa informação. É o responsável pela primeira tomada de decisão do fluxo.
- os nós de respostas específicas: são os nós de processamento final. Esperam até que o roteador defina o destino e, quando o fluxo chega neles, eles executam a cadeia específica (`chain_beach` ou `chain_mountain`) para gerar a resposta baseada na consulta do usuário.

Essas funções se comunicam através do "estado". O conceito de **estado** é o coração do LangGraph, ele representa o _schema_ de dados imutável que encapsula o contexto global de execução compartilhado entre os nós de um grafo. Ele funciona como uma estrutura de dados de controle que armazena, atualiza e propaga as informações necessárias (_inputs_, saídas intermediárias e resultados finais) para que os componentes do sistema operem de forma consistente e encadeada, permitindo que a máquina de estados determine o fluxo de transição entre os nós com base nos valores contidos nesse objeto. Todos os nós do grafo precisam prencher e consultar o "estado".

Ou seja, o estado funciona exatamente como o meio de comunicação e a fonte de verdade compartilhada entre os nós. Como cada nó não chama o próximo diretamente, eles se comunicam de forma indireta: um nó processa uma informação e a escreve no objeto de estado, e o nó seguinte extrai essa informação atualizada para realizar sua própria tarefa, garantindo a coordenação do fluxo sem que haja acoplamento direto entre as funções.

Abaixo definimos o estado através da classe `State`, utilizando `TypedDict` e contendo três campos:

- `query` (a pergunta do usuário)
- `destination` (a rota decidida pelo roteador)
- `response` (o texto final). . Em nossa implementação, ele contém 

In [9]:
# define o "estado": schema do contexto global da execucao
class State(TypedDict):
    query:       str
    destination: Route
    response:    str

A classe `State` é fundamental devido a:

1. **Memória de trânsito:** quando um nó, decide que o destino é "praia", ele não guarda essa informação apenas para si, ele a escreve no estado. Quando o grafo for decidir para onde enviar o fluxo, o campo `destination` do estado será consultado. Sem o estado, os nós seriam "cegos" e não saberiam o que foi processado anteriormente.
2. **Imutabilidade e Segurança:** ao definir o estado com `TypedDict`, estamos estabelecendo um contrato, de que qualquer nó que fizer parte do fluxo precisa lidar com esses campos específicos. Isso evita que informações se percam ou que um nó tente acessar dados que ainda não existem.
3. **Ponto de convergência:** o estado é o único ponto de comunicação entre os nós, que devem ser independentes. Um nó não precisa saber como outro funciona, ele só precisa saber que, ao ser chamado, o estado já contém a a informação necessária para ele executar sua tarefa.

E agora voltando as funções que representam os nós do grafo. Cada função vai disparar a execução de uma cadeia, mas como a execução deve ser assíncrona, vamos passar a usar o método [`ainvoke`](https://reference.langchain.com/python/langgraph/pregel/main/Pregel/ainvoke).

Além disso, também vamos passar o parâmetro `config`, que é uma instância de [`RunnableConfig`](https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig) e atua como um objeto de contexto e controle para a execução dos componentes. Diferente de `state`, que armazena dados do negócio (como perguntas e respostas) e contém _"o que"_ estamos processando, o `config` armazena metadados de execução e configurações de _runtime_ e contém o _"como"_ estamos processando.

In [10]:
from langchain_core.runnables import RunnableConfig

# define funcao para nó roteador, acessa a pergunta do usuario do estado (query)
async def node_router(state: State, config: RunnableConfig):
    return {"destination": await chain_router.ainvoke({"query": state["query"]}, config)}

# define funcoes para nós das cadeias de resposta especifica
async def node_beach(state: State, config=RunnableConfig):
    return {"response": await chain_beach.ainvoke({"query": state["query"]}, config)}

async def node_mountain(state: State, config=RunnableConfig):
    return {"response": await chain_mountain.ainvoke({"query": state["query"]}, config)}

# define funcao que escolhe proximo nó de acordo com estado
def chose_node(state: State) -> Literal["praia", "montanha"]:
    # print(state)
    return "praia" if state["destination"]["destination"] == "praia" else "montanha"

E agora montamos o grafo, definindo sua topologia e regras de navegação. Primeiro inicializamos o grafo instânciando um objeto da classe [`StateGraph`](https://reference.langchain.com/python/langgraph/graph/state/StateGraph) e definindo que a "troca de mensagens" entre os nós deve seguir a estrutura da classe `State`. 

Em seguida adicionamos os nós usando [`ad_node`](https://reference.langchain.com/python/langgraph/graph/state/StateGraph/add_node). Para cada nó adicionado passamos dois argumentos: o identificador do nó (_string_) e a função assíncrona (_callable_) que será executada quando o fluxo chegar nesse nó.

E por fim definimos as "arestas", que são as regras sobre como navegar entre os nós. Aqui temos duas funções que podemos usar: [`add_edge`](https://reference.langchain.com/python/langgraph/graph/state/StateGraph/add_edge) serve para definir um caminho fixo baseado nos identificadores dos nós, sendo o primeiro parâmetro o inicio (um único identificador ou lista de identificadores); e [`add_conditional_edges`](https://reference.langchain.com/python/langgraph/graph/state/StateGraph/add_conditional_edges), que usamos para criar ramificações dinâmicas, passando o nó de início e uma função que retorno o identificador do nó seguinte.

Note que os identificadores dos nós, classes e valores retornados por funções "roteadoras", como `chose_node`, devem ser consistentes entre si quanto aos valores que indetificam nós/componentes.

In [11]:
from langgraph.graph import StateGraph, START, END

# inicializa grafo e define estrutura do estado que deve ser usado
graph = StateGraph(State)

# adiciona os nós do grafo: define o que sera executado em cada nó
graph.add_node(node="routing",  action=node_router)
graph.add_node(node="praia",    action=node_beach)
graph.add_node(node="montanha", action=node_mountain)

# adiciona arestas: regras para navegar entre os nós
graph.add_edge(START, "routing")
graph.add_conditional_edges("routing", chose_node)
graph.add_edge("praia", END)
graph.add_edge("montanha", END)

O fluxo definido acima segue uma lógica de máquina de estados:

1. **Início (START):** o grafo é disparado e o primeiro nó executado é `routing`.
2. **Processamento no roteador:** a função `node_router` é executada. Ela consulta o modelo de linguagem para determinar se a pergunta é sobre _**"praia"**_ ou _**"montanha"**_ e atualiza o objeto estado (`State`) com essa informação.
3. **Decisão:** após o nó `routing` terminar, o grafo chama a função `chose_node`. Ela consulta o estado e decide o próximo nó a receber o fluxo: `'praia'` ou para o nó `'montanha'`".
4. **Execução Final:** o nó escolhido é executado, processa a pergunta específica daquele tema e atualiza o estado com a resposta.
5. **Encerramento:** após a execução do nó de destino, a aresta `add_edge(..., END)` é seguida, indicando que o processamento está concluído e o resultado final pode ser retornado para quem chamou a aplicação.

Note que, anteriormente tinhamos um fluxo direto. Agora, o código se baseia na transição entre estados: o sistema "entende" em qual parte do processo ele está, o que ele já sabe e o que ele ainda precisa descobrir. Essa estrutura permite montar lógicas complexas e ramificadas apenas conectando pontos, sem precisar de uma sequência enorme de comandos `if/else` espalhados pelo código principal.

Finalmente vamos "dar vida" ao grafo e executar a aplicação na prática:

In [12]:
import asyncio

# compila o grafo definido e gera objeto executavel
app = graph.compile()

async def main():
    response = await app.ainvoke({"query": "Quero visitar um lugar no Brasil famoso por praias e cultura"})
    # print(response)
    print(response["response"])

# asyncio.run(main()) # RuntimeError: asyncio.run() cannot be called from a running event loop

Na célula acima é feita:

- Compilação do grafo: transforma a estrutura lógica definida (nós e arestas) em um objeto executável (`app`). É nesse momento que o LangGraph valida se o grafo está completo e pronto para processar dados.
- Definição do ponto de entrada: definimos uma função assíncrona para executar o grafo, pois o LangGraph foi desenhado para lidar com operações de rede (como consultas a modelos de IA) que possuem tempo de espera, então precisamos trabalhar dentro desse contexto assíncrono para não travar a execução do seu programa. Dentro dessa função implementamos o disparo da execução do grafo, enviando o estado inicial apenas com o campo `'query'`. O método `ainvoke` inicia o fluxo de trabalho começando no nó `START`. Quando o grafo termina e chega em algum nó `END`, é retornado o estado final atualizado.
- Inicia execução: o comando `run` da biblioteca [`asyncio`](https://docs.python.org/3/library/asyncio.html) é o que efetivamente inicia o _loop_ de eventos do Python ao executar a sua função `main`.

Uma particularidade que temos nesse ambiente é que, como o Jupyter Notebook já roda seu próprio _loop_ de eventos, o comando `asyncio.run()` tenta criar um novo loop do zero, mas ele detecta que um já está ativo e, por segurança, bloqueia a execução para evitar conflitos. Então nesse caso basta rodar:

```python
await main()
```

Ou ainda, podemos invocar o grafo diretamente sem precisar envolver em outra função:

```python
resposta = await app.ainvoke({"query": "Quero visitar um lugar no Brasil famoso por praias e cultura"})
print(resposta["resposta"])
```

In [13]:
response = await app.ainvoke({"query": "Quero visitar um lugar no Brasil famoso por praias e cultura"})
print(response["response"])

Olá! Sou a **Sra. Praia**, sua especialista em viagens litorâneas. 🌊

Para unir praias e cultura, minha recomendação é **Salvador, na Bahia**. Você pode conhecer o Pelourinho, ouvir música ao vivo, provar acarajé e moqueca e ainda aproveitar praias como Porto da Barra e Itapuã.

Se quiser, também posso sugerir um roteiro de alguns dias — ou comparar Salvador com Recife e Olinda.


In [14]:
response

{'query': 'Quero visitar um lugar no Brasil famoso por praias e cultura',
 'destination': {'destination': 'praia'},
 'response': 'Olá! Sou a **Sra. Praia**, sua especialista em viagens litorâneas. 🌊\n\nPara unir praias e cultura, minha recomendação é **Salvador, na Bahia**. Você pode conhecer o Pelourinho, ouvir música ao vivo, provar acarajé e moqueca e ainda aproveitar praias como Porto da Barra e Itapuã.\n\nSe quiser, também posso sugerir um roteiro de alguns dias — ou comparar Salvador com Recife e Olinda.'}

In [15]:
response = await app.ainvoke({"query": "Quero fazer um passeio radical na natureza"})
print(response["response"])

Olá! Sou o Sr. Montanha, especialista em viagens de aventura na natureza. 🏔️

Que tipo de emoção você procura: trilha com cachoeira, rafting, rapel, escalada ou tirolesa? E em que cidade ou região você gostaria de fazer o passeio? Assim posso sugerir opções que combinem com seu nível de experiência.


In [16]:
response

{'query': 'Quero fazer um passeio radical na natureza',
 'destination': {'destination': 'montanha'},
 'response': 'Olá! Sou o Sr. Montanha, especialista em viagens de aventura na natureza. 🏔️\n\nQue tipo de emoção você procura: trilha com cachoeira, rafting, rapel, escalada ou tirolesa? E em que cidade ou região você gostaria de fazer o passeio? Assim posso sugerir opções que combinem com seu nível de experiência.'}